***EDA Hospitales***

No se filtrará por ningún valor y no hay valores nulos en campos principales.

Los campos de tipo string se debe limpiar los caracteres raros como "".

Las coordenadas estan metros se debe pasar a grados decimales para poder graficar. Usar `st_transform`.

Las coordenadas son válidas, estan dentro de CABA.


In [0]:
%sql

describe proyecto_final.raw.hospitales_bronze;

In [0]:
%sql

select * from 
proyecto_final.raw.hospitales_bronze
limit 36;

In [0]:
%sql

--Nulos

select 
count(*) as total,
count(*) - count(nombre_hospital) as nombre_hospital_nulos,
count(*) - count(nombre_funcional_completo) as nombre_funcional_completo_nulos,
count(*) - count(nombre_generico) as nombre_generico_nulos,
count(*) - count(especialidad) as especialidad_nulos,
count(*) - count(tipo_atencion) as tipo_atencion_nulos,
count(*) - count(direccion) as direccion_nulos,
count(*) - count(barrio) as barrio_nulos,
count(*) - count(comuna) as comuna_nulos,
count(*) - count(telefono) as telefono_nulos,
count(*) - count(sitio_web) as sitio_web_nulos,
count(*) - count(fuente_datos) as fuente_datos_nulos,
count(*) - count(geometry) as geometry_nulos
from proyecto_final.raw.hospitales_bronze

In [0]:
%sql
--duplicados

select 
nombre_hospital,
geometry,
count(*)
from proyecto_final.raw.hospitales_bronze
group by nombre_hospital,geometry
having count(*) >1

In [0]:
%sql
--19 barrios distintos. Mayor cantidad en parque patricios
select barrio, count(*) cantidad
from proyecto_final.raw.hospitales_bronze
group by barrio
order by cantidad desc

In [0]:
%sql
-- Distribucion de comunas
select comuna, count(*)
from proyecto_final.raw.hospitales_bronze
group by comuna
order by 2 desc

In [0]:
%sql 

select especialidad,
count(*) cantidad,
round(count(*)*100/sum(count(*)) over(),2) as porcentaje
from proyecto_final.raw.hospitales_bronze
group by especialidad
order by cantidad desc

In [0]:
select * from proyecto_final.raw.hospitales_bronze
where especialidad is null

-- No es un registro especial, el resto de los campos tiene valores

In [0]:
%sql
--Duplicados. No hay

select nombre_funcional_completo,
geometry,
count(*) 
from proyecto_final.raw.hospitales_bronze
group by nombre_funcional_completo,geometry
having count(*) >1

In [0]:
SELECT st_astext(st_transform(st_geomfromtext(geometry, 9498), 4326)) coordenadas, -- long lat,
st_x(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS longitud,
st_y(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS latitud
FROM proyecto_final.raw.hospitales_bronze

In [0]:
%sql
-- Si las coordenadas esta dentro de CABA
with coordenadas as (
  SELECT st_astext(st_transform(st_geomfromtext(geometry, 9498), 4326)) coordenadas, -- long lat,
  st_x(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS longitud,
  st_y(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS latitud
  FROM proyecto_final.raw.hospitales_bronze
)
select * from coordenadas
WHERE latitud > -34.50 OR latitud < -34.70 
   OR longitud > -58.30 OR longitud < -58.55;

In [0]:
%sql
-- No hay carateres raros pero igual limpiare los campos 
--Geometry esta con coordenadas en metros. Se debe pasar a latitud y longitud en grados 

In [0]:
%sql

CREATE OR REPLACE VIEW v_hospitales_limpieza AS
SELECT 
    -- 1. Limpieza de nombres y categorías: minúsculas y quitar comillas dobles
    LOWER(REPLACE(TRIM(nombre_hospital), '"', '')) AS nombre_hospital,
    LOWER(REPLACE(TRIM(nombre_funcional_completo), '"', '')) AS nombre_funcional_completo,
    LOWER(REPLACE(TRIM(nombre_generico), '"', '')) AS nombre_generico,
    LOWER(REPLACE(TRIM(especialidad), '"', '')) AS especialidad,
    LOWER(REPLACE(TRIM(tipo_atencion), '"', '')) AS tipo_atencion,
    LOWER(REPLACE(TRIM(direccion), '"', '')) AS direccion,
    LOWER(REPLACE(TRIM(barrio), '"', '')) AS barrio,
    LOWER(REPLACE(TRIM(fuente_datos), '"', '')) AS fuente_datos,
    
    -- 2. Identificadores numéricos (sin cambios)
    comuna,
    
    -- 3. Datos de contacto (se limpian comillas y espacios, pero se mantiene el casing original)
    REPLACE(TRIM(telefono), '"', '') AS telefono,
    REPLACE(TRIM(sitio_web), '"', '') AS sitio_web,

    -- 4. Geometría y auditoría
    geometry,
    fecha_ingesta,
    st_x(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS longitud,
   st_y(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS latitud
FROM proyecto_final.raw.hospitales_bronze
-- 5. Filtro de calidad para el EDA
WHERE nombre_hospital IS NOT NULL 
  AND geometry IS NOT NULL;

In [0]:
select * from v_hospitales_limpieza